# 05b - HayFlow-Hines corrected overfit canary

This notebook repeats only the architectural canary. It does not launch full training. The acceptance thresholds are identical to notebook 05; objective staging, boundary spike representation, checkpoint selection and diagnostics are corrected.

## 1. Coherent checkout and runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml'], check=True)
sys.path.insert(0, str(ELM_REPO))
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle prima di eseguire il canary.'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)})

## 2. Composite dataset
Only the complete targeted v1.1 base dataset and BAP top-up v3 are required. Notebook 04/B3 is deliberately not an input to this diagnostic.

In [ ]:
import shutil, zipfile
INPUT_ROOT = Path('/kaggle/input')

def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination)
    marker = destination / '.source_size'
    stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp:
        return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True)
    root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp)
    return destination

topup_override = os.environ.get('HAYFLOW_TOPUP_V3')
topup_candidates = [Path(topup_override).expanduser()] if topup_override else []
topup_candidates.extend(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))
topup_candidates.extend(path.parent for path in INPUT_ROOT.rglob('composite_dataset_manifest.json'))
TOPUP_SOURCE = next((p.resolve() for p in topup_candidates if p.exists()), None)
assert TOPUP_SOURCE is not None, 'Top-up BAP v3 non trovato negli input Kaggle.'
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05b_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'))
assert len(manifest_candidates) == 1, manifest_candidates
COMPOSITE_MANIFEST = manifest_candidates[0]

base_override = os.environ.get('HAYFLOW_BASE_DATASET')
base_candidates = [Path(base_override).expanduser()] if base_override else []
base_candidates.extend(path.parent for path in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(path).lower() and 'topup' not in str(path).lower())
base_candidates.extend(path for path in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(path).lower())
BASE_SOURCE = next((p.resolve() for p in base_candidates if p.exists()), None)
assert BASE_SOURCE is not None, 'Dataset base targeted v1.1 non trovato.'
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE)})

## 3. Cryptographic preflight and normalization

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle

hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now)
    percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9)
        eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05b][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True)
        hash_last[name] = percent

bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
display(bundle.report)
assert bundle.report['valid']
assert bundle.report['episode_count'] == 369
assert bundle.report['transition_count'] == 29880
assert bundle.report['topup_validation_only']

In [ ]:
from src.hayflow_model import HayFlowHinesExperiment, HinesPrototypeExperimentConfig
raw = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_canary_v2.yml').read_text())
config = HinesPrototypeExperimentConfig.from_mapping(raw['experiment'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_canary_v2')
session = HayFlowHinesExperiment(bundle, OUTPUT_DIR, config)
prepare_report = session.prepare()
display(prepare_report)
assert prepare_report['training_contract_blockers'] == []

## 4. Independent Hines checks

In [ ]:
hines_report = session.run_hines_layer_tests()
display(hines_report)
assert hines_report['valid']

## 5. Corrected staged canary
This is the long cell. Each model performs 120 voltage/peak epochs, 120 event/branching epochs and 60 joint epochs. The best checkpoint is selected by the four acceptance metrics, not total training loss.

In [ ]:
canary_report = session.run_canary()
display({
    'scenario': canary_report['scenario'],
    'proceed_to_full_training': canary_report['proceed_to_full_training'],
    'event_support': canary_report['event_support'],
    'curriculum': canary_report['curriculum'],
    'models': {name: {key: value for key, value in report.items() if key != 'history'} for name, report in canary_report['models'].items()},
})
print('05b terminato. Non avviare training completo in questo notebook.')

## 6. Output contract

In [ ]:
required = [
    OUTPUT_DIR / 'canary_overfit_report.json',
    OUTPUT_DIR / 'model_configurations.json',
    OUTPUT_DIR / 'normalization_schema.json',
    OUTPUT_DIR / 'hines_layer_tests.json',
    OUTPUT_DIR / 'checkpoints' / 'canary_models.pt',
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, missing
print({'output_dir': str(OUTPUT_DIR), 'files': len(list(OUTPUT_DIR.rglob('*'))), 'missing': missing})

## 7. Browser download
The ZIP includes the canary checkpoints. The download is generated as a browser Blob from Base64 because this is the reliable Kaggle method for this project.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, display

zip_base = Path('/kaggle/working/hayflow_hines_canary_v2')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
filename = zip_path.name
display(Javascript(f'''
const binary = atob('{encoded}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
'''))
print('Download avviato:', filename, f'({zip_path.stat().st_size / 2**20:.1f} MiB)')